# Demo: Building a RAG-powered FAQ Agent with Custom Knowledge

# Step 1: Install required packages

In [1]:
# Install all of the relvant tools for agent building 
!pip install -U langchain langchain-openai langchain-community langchain-classic faiss-cpu tiktoken

# Step 2: Import dependencies

In [2]:
import os
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter# for chunking 
from langchain_community.vectorstores import FAISS
# vectorstores is for storage purposes  # computers only understand numbers hence vectorize the text, called embeddings store in vector db 
# FAISS is one of the classes helping to store the embeddings in a vector db 
from langchain_classic.chains import RetrievalQA  # helping to load some documets , load some embeddings 
#update

/tmp/ipykernel_15844/925807929.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


# Step 3: Set Azure OpenAI credentials



In [3]:
os.environ["AZURE_OPENAI_API_KEY"] = "2ABecnfxzhRg4M5D6pBKiqxXVhmGB2WvQ0aYKkbTCPsj0JLKsZPfJQQJ99BDAC77bzfXJ3w3AAABACOGi3sC"# same clone of the lab sessions 

# Step 4: Load and chunk your custom FAQ document


In [4]:
# Load the FAQ document and split it into chunks for embedding
loader = TextLoader("faq.txt")  # Ensure this file exists
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)# how do we define the chunk size ? overlap is needed to retain context across chunks 
docs = text_splitter.split_documents(documents)
# Check if the loader is initialized correctly 

# Step 5: Create vectorstore using Azure embeddings


In [5]:
# Embed document chunks and store them in a FAISS vector index
embeddings = AzureOpenAIEmbeddings(# numerical conversions of texts 
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    api_version="2023-05-15",
    deployment="text-embedding-ada-002",
    api_key=os.environ["AZURE_OPENAI_API_KEY"]
)

vectorstore = FAISS.from_documents(docs, embeddings)

# Step 6: Initialize the Azure OpenAI LLM

In [6]:
# Initialize the GPT-4o model from Azure with temperature 0 for deterministic output

llm = AzureChatOpenAI(
    azure_endpoint="https://openai-api-management-gw.azure-api.net",
    api_version="2025-01-01-preview",
    deployment_name="gpt-5-mini"# LLM detail using gpt 5 mini 
)
#update

# Step 7: Create the RAG chain

### Why `return_source_documents=True` matters

This tells the RAG chain to return the actual chunks retrieved from the vector store alongside the final answer.

- `result["result"]` = the answer generated by the model
- `result["source_documents"]` = the documents/chunks used as context

This helps us verify that the answer is grounded in the retrieved knowledge and makes it easier to inspect the sources used for the response.

In [8]:
# Create a RetrievalQA chain that uses the retriever and LLM to answer queries with source context

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

In [ ]:
# # Demo: show how the retriever can source exactly 2 documents
# from langchain_core.documents import Document

# # Two source documents that are relevant to the same question
# source_docs = [
#     Document(page_content="Our refund policy allows returns within 30 days for unused products and a full refund to the original payment method."),
#     Document(page_content="Customer support can help with billing questions, technical issues, and account access 24/7 through email or live chat."),
# ]

# # Create a tiny vector store using only these two documents
# mini_vectorstore = FAISS.from_documents(source_docs, embeddings)
# retriever = mini_vectorstore.as_retriever(search_kwargs={"k": 2})

# query = "What support options and refund policy do you offer?"
# retrieved_docs = retriever.invoke(query)

# print(f"Retrieved {len(retrieved_docs)} documents:")
# for i, doc in enumerate(retrieved_docs, 1):
#     print(f"\nDocument {i}: {doc.page_content}")


Retrieved 2 documents:

Document 1: Our refund policy allows returns within 30 days for unused products and a full refund to the original payment method.

Document 2: Customer support can help with billing questions, technical issues, and account access 24/7 through email or live chat.


# Step 8: Ask a question

In [ ]:
# Send a question to the RAG chain and store the result
query = "What is our return policy?"
result = qa_chain.invoke({"query": query})

# Step 9: Print results

In [16]:
# Display the final answer and the source chunks used
query = "What is the capital of London?"
result = qa_chain.invoke({"query": query})

print("Answer:", result["result"])
print("\n--- Sources ---")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i}:")
    print(doc.page_content)

Answer: London doesn't have a capital — London itself is a capital city. It is the capital of England and of the United Kingdom. Did you mean something else (for example, an administrative center within Greater London)?

--- Sources ---

Source 1:
Q: What is your return policy?
A: We offer a 30-day return policy for all unopened items. Products must be returned in original packaging with proof of purchase.

Q: Do you offer international shipping?
A: Yes, we ship internationally to over 50 countries. Delivery times and shipping costs vary by destination.

Q: How can I contact customer support?
A: You can reach us via email at support@example.com or call our helpline at +1-800-123-4567. Our support hours are 9AM to 6PM, Monday to Friday.

Source 2:
Q: Do you offer gift wrapping?
A: Yes, gift wrapping can be added at checkout for an additional ₹50 per item.

Source 3:
Q: What payment methods are accepted?
A: We accept Visa, MasterCard, American Express, PayPal, and UPI payments.

Q: How c